# Data Extraction & Profiling Pipeline
This notebook walks through building a basic data pipeline —  
from creating a dataset, loading it, profiling it, and logging what happened.

---
## Segment 1 — Imports

Before doing anything, we bring in the libraries we need.

- **`pandas`** — handles our data (think of it as Excel inside Python)
- **`json`** — lets us read and write `.json` files (used later for logging)
- **`os`** — helps us interact with the file system (create folders, check paths, etc.)

In [ ]:
import pandas as pd
import json
import os

---
## Segment 2 — Setting Up Folders

The CSV and log files need folders to live in.  
We create them here if they don't already exist — `exist_ok=True` means it won't crash if the folder is already there.

In [ ]:
os.makedirs('data/raw', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("Folders ready.")

---
## Segment 3 — The Logging Function

We define a helper function and an empty log list **before** anything else,  
because later steps will call this function to record what happened.

Every time `log_transformation()` is called, it appends a small record  
(step name + message) to `transformation_log`.

In [ ]:
transformation_log = []

def log_transformation(step_name, message):
    entry = {
        "step": step_name,
        "message": message
    }
    transformation_log.append(entry)
    print(f"[LOG] {step_name}: {message}")

---
## Segment 4 — Creating a Sample Dataset

Since we don't have a real dataset, we build one from scratch with 100 rows.

| Column | What it contains |
|---|---|
| `id` | Numbers 1 to 100 |
| `value` | Each number × 1.5 (so 0.0, 1.5, 3.0 ...) |
| `category` | Cycles through A, B, C, and `None` (missing value on purpose) |
| `timestamp` | One date per row, starting Jan 1 2023 |

> `to_csv(..., index=False)` saves it without adding an extra unwanted row-number column.

In [ ]:
raw_data_path = 'data/raw/sample_data.csv'

sample_df = pd.DataFrame({
    'id': range(1, 101),
    'value': [x * 1.5 for x in range(100)],
    'category': ['A', 'B', 'C', None] * 25,
    'timestamp': pd.date_range(start='2023-01-01', periods=100, freq='D')
})

sample_df.to_csv(raw_data_path, index=False)
print(f"Sample data created at: {raw_data_path}")

# Quick peek at what we just made
sample_df.head()

---
## Segment 5 — Loading the CSV

Now we reload the file we just saved — this simulates the real-world scenario  
where someone hands you a CSV file and you load it fresh.

In [ ]:
df = pd.read_csv(raw_data_path)

print(f"Loaded {len(df)} rows and {len(df.columns)} columns")
df.head()

---
## Segment 6 — Profiling the Data

Before cleaning or transforming anything, we **understand** what we're working with.

- `isnull().sum()` → counts missing values per column
- `dtypes` → shows whether each column is a number, string, date, etc.
- `describe(include='all')` → gives stats like mean, min, max — the `include='all'` part forces it to include text columns too (otherwise it skips them)

In [ ]:
profiling_stats = {
    "row_count": len(df),
    "column_count": len(df.columns),
    "missing_values": df.isnull().sum().to_dict(),
    "data_types": df.dtypes.astype(str).to_dict(),
    "summary_stats": df.describe(include='all').to_dict()
}

print("--- Profiling Summary ---")
print(f"Columns     : {list(df.columns)}")
print(f"Row count   : {profiling_stats['row_count']}")
print(f"Nulls found : {profiling_stats['missing_values']}")
print(f"Data types  : {profiling_stats['data_types']}")

---
## Segment 7 — Logging the Profiling Step

We call our `log_transformation()` function to record that profiling happened  
and what it found. This is useful in real pipelines — you can look back at the log  
and know exactly what each step did.

In [ ]:
log_transformation(
    "Data Extraction & Profiling",
    f"Loaded {len(df)} rows. Found missing values: {profiling_stats['missing_values']}."
)

---
## Segment 8 — Saving the Log to a File

Finally, we write the full log out to a `.json` file.

- `'w'` mode overwrites the file each run (clean slate every time)
- `indent=4` makes the JSON readable with proper spacing instead of one giant line

In [ ]:
with open('logs/transformation_log.json', 'w') as f:
    json.dump(transformation_log, f, indent=4)

print("Transformation log saved to: logs/transformation_log.json")
print("\n--- Log Contents ---")
for entry in transformation_log:
    print(f"  Step    : {entry['step']}")
    print(f"  Message : {entry['message']}")

---
## Summary

Here's what this notebook does end to end:

```
Create folders → Define logger → Make fake data → Save CSV
     → Reload CSV → Profile it → Log results → Save log file
```

This is the skeleton of a real **ETL pipeline** (Extract, Transform, Load) —  
the kind of structure used in data engineering and analytics projects.